# 01. KNN и логистическая регрессия

## Только классификация

Задача этого ноутбука — понять две разные идеи построения классификатора:

1. **K ближайших соседей (KNN)** — решение принимается по объектам, которые находятся рядом с новым объектом.
2. **Логистическая регрессия** — строится математическая модель вероятности принадлежности объекта к классу.

Особенно подробно разберём, почему для логистической регрессии естественно появляется **Log Loss**:  
**распределение Бернулли → функция правдоподобия → логарифм правдоподобия → минимизация Log Loss**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

sns.set_theme(style="whitegrid")

## 1. KNN — метод ближайших соседей

**KNN (K-Nearest Neighbors)** — один из самых простых алгоритмов классификации.

Главная идея:

> Чтобы определить класс нового объекта, посмотрим на его `k` ближайших обучающих объектов и выберем класс, который встречается среди них чаще всего.

KNN называют **методом, основанным на примерах**: в классическом варианте он практически не строит явную модель во время обучения. Он запоминает обучающую выборку, а основная работа выполняется при прогнозировании.

### Как работает KNN

Для нового объекта выполняются следующие шаги:

1. Выбираем число соседей `k`.
2. Вычисляем расстояние от нового объекта до обучающих объектов.
3. Выбираем `k` объектов с наименьшим расстоянием.
4. Смотрим на их классы.
5. Для классификации выбираем класс большинства.

Например, при `k=5` четыре ближайших объекта имеют класс `1`, а один — класс `0`. Тогда прогноз будет `1`.

Важное следствие: **масштаб признаков имеет значение**. Если один признак измеряется в тысячах, а другой — в единицах, расстояние может определяться почти исключительно первым признаком. Поэтому перед KNN часто применяют стандартизацию.

In [ ]:
X, y = make_moons(n_samples=250, noise=0.22, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Класс 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Класс 1")
plt.xlabel("X1")
plt.ylabel("X2")
plt.title("Данные для классификации")
plt.legend()
plt.show()

### Гиперпараметры KNN

Основные гиперпараметры:

- `n_neighbors` — количество соседей `k`;
- `weights` — как учитывать соседей: одинаково (`uniform`) или сильнее учитывать близких (`distance`);
- `metric` — функция расстояния, например евклидово расстояние;
- `p` — параметр расстояния Минковского;
- `algorithm` — способ поиска ближайших соседей.

#### Что происходит при изменении `k`

- Маленькое `k` → модель чувствительна к шуму и может переобучаться.
- Большое `k` → граница становится более гладкой, но модель может недообучаться.

То есть `k` управляет компромиссом между гибкостью и обобщающей способностью.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

for k in [1, 5, 15]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    print(f"k={k}: accuracy={accuracy_score(y_test, model.predict(X_test)):.3f}")

### Визуализация границы KNN

В двумерной задаче можно окрасить всю плоскость в соответствии с предсказанным классом. Полученная линия/область показывает, **где алгоритм меняет решение с класса 0 на класс 1**.

In [ ]:
def plot_boundary(model, X, y, title):
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min,x_max,400),
                         np.linspace(y_min,y_max,400))
    grid = np.c_[xx.ravel(), yy.ravel()]
    z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8,6))
    plt.contourf(xx, yy, z, alpha=0.20)
    plt.scatter(X[y==0,0], X[y==0,1], label="Класс 0", edgecolor="black")
    plt.scatter(X[y==1,0], X[y==1,1], label="Класс 1", edgecolor="black")
    plt.xlabel("X1"); plt.ylabel("X2"); plt.title(title); plt.legend(); plt.show()

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])
knn.fit(X_train, y_train)
plot_boundary(knn, X, y, "Граница классификации KNN, k=7")

### Псевдоалгоритм KNN на Python

Ниже не готовая промышленная реализация, а упрощённая схема того, **что алгоритм делает внутри**.

In [ ]:
def pseudo_knn_predict(X_train, y_train, new_object, k):
    # 1. Расстояние от нового объекта до каждого обучающего объекта
    distances = []

    for x, label in zip(X_train, y_train):
        distance = np.sqrt(np.sum((x - new_object) ** 2))
        distances.append((distance, label))

    # 2. Сортируем объекты по расстоянию
    distances.sort(key=lambda item: item[0])

    # 3. Берём k ближайших
    nearest = distances[:k]

    # 4. Голосование классов
    labels = [label for distance, label in nearest]
    classes, counts = np.unique(labels, return_counts=True)

    # 5. Возвращаем класс с максимальным числом голосов
    return classes[np.argmax(counts)]

pseudo_knn_predict(X_train, y_train, X_test[0], k=5)

### Достоинства KNN

- Очень простая идея и понятный принцип работы.
- Хорошо подходит для небольших выборок.
- Может строить нелинейные границы классификации.
- Не требует предполагать конкретную форму разделяющей границы.

### Недостатки KNN

- Предсказание может быть дорогим на больших датасетах: приходится искать соседей.
- Чувствителен к масштабу признаков.
- Чувствителен к нерелевантным признакам и шуму.
- При большой размерности расстояния становятся менее информативными — проблема, связанная с **проклятием размерности**.
- Требуется подобрать `k` и способ вычисления расстояния.

# 2. Логистическая регрессия

Несмотря на название, в этом ноутбуке логистическая регрессия рассматривается **только как алгоритм классификации**.

Для бинарной классификации нам нужно получить вероятность:

$$
P(y=1\mid x).
$$

Логистическая регрессия сначала вычисляет линейную комбинацию признаков:

$$
z = w_0+w_1x_1+...+w_px_p,
$$

а затем преобразует её в число от 0 до 1 с помощью сигмоиды:

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Если вероятность выше выбранного порога, например `0.5`, прогнозируем класс 1, иначе класс 0.

In [ ]:
z = np.linspace(-7, 7, 300)
p = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8,5))
plt.plot(z, p)
plt.axhline(0.5, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("z = w0 + w1*x1 + ... + wp*xp")
plt.ylabel("P(y=1)")
plt.title("Сигмоидальная функция")
plt.show()

## Почему используется Log Loss?

Это ключевой момент.

### Шаг 1. Распределение Бернулли

В бинарной классификации целевая переменная принимает два значения:

$$
y\in\{0,1\}.
$$

Если модель предсказывает вероятность $p=P(y=1\mid x)$, то вероятность наблюдаемого значения `y` можно записать одной формулой:

$$
P(y\mid x)=p^y(1-p)^{1-y}.
$$

Это и есть распределение Бернулли.

### Шаг 2. Метод максимального правдоподобия

Пусть у нас есть обучающая выборка из `n` объектов. Предполагая независимость наблюдений, её правдоподобие:

$$
L(w)=\prod_{i=1}^{n}p_i^{y_i}(1-p_i)^{1-y_i}.
$$

Мы хотим подобрать параметры `w`, при которых наблюдаемая выборка наиболее вероятна:

$$
\max_w L(w).
$$

### Шаг 3. Берём логарифм

Произведение неудобно вычислять. Логарифм превращает произведение в сумму:

$$
\log L(w)=\sum_{i=1}^{n}
[y_i\log p_i+(1-y_i)\log(1-p_i)].
$$

Логарифм монотонно возрастает, поэтому максимизация `L` эквивалентна максимизации `log L`.

### Шаг 4. Переходим к минимизации

В машинном обучении обычно минимизируют функцию потерь. Поэтому меняем знак:

$$
\boxed{
LogLoss=-\frac{1}{n}\sum_{i=1}^{n}
[y_i\log p_i+(1-y_i)\log(1-p_i)]
}
$$

Таким образом:

**Бернулли → правдоподобие → log likelihood → смена знака → Log Loss.**

Именно поэтому Log Loss естественным образом возникает для бинарной логистической классификации.

In [ ]:
p = np.linspace(0.001, 0.999, 500)

loss_if_y1 = -np.log(p)
loss_if_y0 = -np.log(1-p)

plt.figure(figsize=(8,5))
plt.plot(p, loss_if_y1, label="Истинный класс y=1")
plt.plot(p, loss_if_y0, label="Истинный класс y=0")
plt.xlabel("Предсказанная вероятность P(y=1)")
plt.ylabel("Log Loss")
plt.title("Почему уверенная неправильная вероятность сильно штрафуется")
plt.ylim(0, 8)
plt.legend()
plt.show()

### Интуиция Log Loss

Если истинный класс `y=1`:

- прогноз `p=0.9` → маленькая потеря;
- прогноз `p=0.6` → потеря больше;
- прогноз `p=0.01` → огромная потеря.

То есть Log Loss оценивает не только правильность класса, но и **качество вероятностного прогноза**.

### Гиперпараметры логистической регрессии

Основные параметры `LogisticRegression`:

- `C` — обратная сила регуляризации. Маленькое `C` означает более сильную регуляризацию.
- `penalty` — тип регуляризации, например L2 или L1 в поддерживаемых solver.
- `solver` — численный алгоритм оптимизации.
- `class_weight` — веса классов; особенно полезен при дисбалансе.
- `max_iter` — максимальное количество итераций оптимизации.

Регуляризация нужна, чтобы ограничивать слишком большие коэффициенты и снижать риск переобучения.

In [ ]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(C=1.0, max_iter=1000))
])

logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("Log Loss:", log_loss(y_test, proba))

In [ ]:
plot_boundary(logreg, X, y, "Граница логистической регрессии")

### Псевдоалгоритм логистической регрессии

Упрощённо обучение можно представить так:

1. Инициализировать коэффициенты `w`.
2. Для каждого объекта вычислить `z = w0 + w1*x1 + ...`.
3. Получить вероятность через сигмоиду.
4. Вычислить Log Loss.
5. Найти направление изменения коэффициентов, уменьшающее loss.
6. Обновить `w`.
7. Повторять до сходимости.

In [ ]:
def pseudo_logistic_model(X, y, learning_rate=0.1, epochs=1000):
    w = np.zeros(X.shape[1])
    b = 0.0

    for _ in range(epochs):
        z = X @ w + b
        p = 1 / (1 + np.exp(-z))

        # Градиент Log Loss
        error = p - y
        dw = X.T @ error / len(X)
        db = np.mean(error)

        w -= learning_rate * dw
        b -= learning_rate * db

    return w, b

### Недостатки логистической регрессии

- Базовая модель строит **линейную границу** в пространстве признаков.
- Для сложных нелинейных зависимостей может потребоваться создание новых признаков или использование другого алгоритма.
- Чувствительна к сильной мультиколлинеарности признаков.
- Выбросы могут влиять на коэффициенты.
- Качество может снижаться, если линейная форма зависимости плохо соответствует задаче.
- Для многоклассовой классификации требуется соответствующая стратегия (например, multinomial или one-vs-rest).

### Сравнение KNN и логистической регрессии

| Свойство | KNN | Логистическая регрессия |
|---|---|---|
| Идея | Соседи | Вероятностная линейная модель |
| Граница | Может быть нелинейной | Линейная |
| Масштабирование | Очень важно | Обычно желательно |
| Обучение | Простое/почти без модели | Оптимизация коэффициентов |
| Предсказание | Может быть дорогим | Обычно быстрое |
| Интерпретируемость | Средняя | Высокая для коэффициентов |
| Вероятности | Зависит от соседей | Естественный вероятностный вывод |

**Главная мысль:** KNN и логистическая регрессия решают одну задачу — классификацию, но используют принципиально разные способы построения решения.